In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("dirty_cafe_sales.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df.isna().sum()

### `Numeric` olan satırlarda işlem yapabilmek için `'ERROR', 'UNKNOWN', np.nan` değerlerini `np.nan` yapıp `pd.to_numeric` ile `float64` değerine çevirelim

### `for c in numeric_cols: df[c] = pd.to_numeric(df[c], errors="coerce")`
### kodu eğer satırlardaki değerler `sayı` ise `olduğu gibi` kabul eder
### satırlardaki değerler `sayı gibi görünüyorsa` "ör : '6'(string)" `numeric değere` çevirir
### satırlardaki değerler `sayı dışında` herhangi bir şey ise `np.nan` olarak çevirir

In [ ]:
# object tipinde olan numeric verileri numeric tipe çevirelim
numeric_cols = ["Price Per Unit", "Total Spent", "Quantity"]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

In [ ]:
df

In [ ]:
df.info()

### `Price Per Unit`sütunundaki bazı `NaN` değerindeki satırları `Total Spent` sütununu `Quantity` sütununa bölerek dolduralım

In [ ]:
df["Price Per Unit"].isna().sum()

In [ ]:
# Price Per Unit sütunundaki eksik değerleri Total Spent ve Quantity sütunlarını kullanarak dolduralım
# maskeleme yaparak eksik Price Per Unit değerlerini hesaplayalım
mask = df["Price Per Unit"].isna() & (df["Quantity"] > 0) & df["Total Spent"].notna()
# eksik Price Per Unit değerlerini hesapla ve ata
df.loc[mask, "Price Per Unit"] = df.loc[mask, "Total Spent"] / df.loc[mask, "Quantity"]

In [ ]:
df["Price Per Unit"].isna().sum()

### `Price Per Unit`sütunundaki bazı `NaN` değerindeki satırları `Item` sütunundaki karşılığa göre dolduralım

In [ ]:
# Price Per Unit sütunundaki eksik değerleri Item sütunundaki bilgilere göre dolduralım
item_price_map = {
    "Cookie" : 1.0,
    "Coffee" : 2.0,
    "Tea" : 1.5,
    "Juice" : 3.0,
    "Cake" : 3.0,
    "Smoothie" : 4.0,
    "Sandwich" : 4.0,
    "Salad" : 5.0
}

# maskeleme yaparak eksik Price Per Unit değerlerini hesaplayalım
mask = df["Price Per Unit"].isna() & df["Item"].isin(item_price_map.keys())
# elsik Price Per Unit değerlerini ismine göre ata
df.loc[mask, "Price Per Unit"] = df.loc[mask, "Item"].map(item_price_map)

In [ ]:
df["Price Per Unit"].isna().sum()

### `Quantity`sütunundaki bazı `NaN` değerindeki satırları `Total Spent` sütununu `Price Per Unit` sütununa bölerek dolduralım

In [ ]:
df["Quantity"].isna().sum()

In [ ]:
# maskeleme yaparak eksik Quantity değerlerini hesaplayalım
mask = df["Quantity"].isna() & (df["Price Per Unit"] > 0) & df["Total Spent"].notna()
# eksik Quantity değerlerini hesapla ve ata
df.loc[mask, "Quantity"] = df.loc[mask, "Total Spent"] / df.loc[mask, "Price Per Unit"]

In [ ]:
df["Quantity"].isna().sum()

### `Total Spent` sütunundaki bazı `NaN` değerindeki satırları `Quantity` sütunu ile `Price Per Unit` satırlarını çarparak dolduralım

In [ ]:
df["Total Spent"].isna().sum()

In [ ]:
# maskeleme yaparak eksik Total Spent değerlerini hesaplayalım
mask = df["Total Spent"].isna() & df["Price Per Unit"].notna() & df["Quantity"].notna()
# eksik Total Spent değerlerini hesapla ve ata
df.loc[mask, "Total Spent"] = df.loc[mask, "Price Per Unit"] * df.loc[mask, "Quantity"]

In [ ]:
df["Total Spent"].isna().sum()

In [ ]:
df.isna().sum()

### `Item` sütunundaki `'ERROR', 'UNKNOWN', np.nan` olan satırları `Price Per Unit` sütununa göre dolduralım

In [ ]:
df[(df["Item"] == "UNKNOWN") | (df["Item"] == "ERROR") | (df["Item"].isna())]

In [ ]:
# Only infer an item when the unit price uniquely identifies it.
items_by_price = {}
for item, price in item_price_map.items():
    items_by_price.setdefault(price, []).append(item)
price_to_item = {price: items[0] for price, items in items_by_price.items() if len(items) == 1}
df['Item'] = df['Item'].replace(['ERROR', 'UNKNOWN'], np.nan)
mask = df['Item'].isna() & df['Price Per Unit'].isin(price_to_item)
df.loc[mask, 'Item'] = df.loc[mask, 'Price Per Unit'].map(price_to_item)
# Ambiguous prices remain missing instead of inventing item identities.

In [ ]:
df[(df["Item"] == "UNKNOWN") | (df["Item"] == "ERROR") | (df["Item"].isna())]

In [ ]:
df.isna().sum()

## Doldurulmayacak bazı satıları veri setimizden kalıcı olarak silelim

In [ ]:
# subset ile sadece Price Per Unit sütunundaki NaN değerine sahip olan satırları düşürelim
df.dropna(subset=["Price Per Unit"], inplace=True)

In [ ]:
# subset ile sadece Quantity sütunundaki NaN değerine sahip olan satırları düşürelim
df.dropna(subset = ["Quantity"], inplace=True)

In [ ]:
df.isna().sum()

In [ ]:
cols = ["Payment Method", "Location"]
problem_values = ["UNKNOWN", "ERROR"]
# Belirtilen sütunlardaki problemli değerleri NaN ile değiştirelim
for c in cols:
    df[c] = df[c].replace(problem_values, np.nan)


In [ ]:
df["Location"].value_counts()

In [ ]:
df["Payment Method"].value_counts()

In [ ]:
# Preserve missing categories instead of inventing a payment or location.
for col in ['Payment Method', 'Location', 'Item']:
    df[col] = df[col].fillna('Unknown')

In [ ]:
# Transaction Date sütununu düşürelim
df.drop(["Transaction Date"], axis=1, inplace=True)

In [ ]:
df.isna().sum()

In [ ]:
df

## Temizlediğimiz .csv dosyasını farklı olarak kaydedelim

In [ ]:
# Keep raw input unchanged; validate the cleaned example before exporting.
df_clean = df.copy()
assert df_clean[['Price Per Unit', 'Quantity', 'Total Spent']].notna().all().all()
print('Nonpositive quantity or price:', ((df_clean['Quantity'] <= 0) | (df_clean['Price Per Unit'] <= 0)).sum())
print('Inconsistent totals:', (~np.isclose(df_clean['Quantity'] * df_clean['Price Per Unit'], df_clean['Total Spent'], rtol=0, atol=0.01)).sum())

In [ ]:
df_clean.to_csv('cleaned_cafe_sales.csv', index=False)